# Análisis Comparativo: YOLOv8s vs YOLOv11s vs YOLOv12s (10 Runs)

**Dataset**: Smart Parking UPeU v4  
**Clases**: libre, no_disponible, ocupado  
**Epochs**: 100 | **Batch**: 16 | **ImgSize**: 640  
**Runs por modelo**: 10 (seeds: 42, 123, 456, 789, 1000, 1234, 2024, 2025, 3141, 9999)

## 4.1. Dataset Composition

### Tabla 1 — Distribución del Dataset

In [4]:
from pathlib import Path
import pandas as pd

dataset_base = Path('/Users/fernando/Desktop/tesis_backup/v2/dataset/ultra')

subsets = {
    'Training':   dataset_base / 'train'  / 'images',
    'Validation': dataset_base / 'valid'  / 'images',
    'Test':       dataset_base / 'test'   / 'images',
}

exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

rows = []
for subset, path in subsets.items():
    count = sum(1 for f in path.iterdir() if f.suffix.lower() in exts) if path.exists() else 0
    rows.append({'Subset': subset, 'Imágenes': count})

total = sum(r['Imágenes'] for r in rows)
rows.append({'Subset': 'Total', 'Imágenes': total})

df_dataset = pd.DataFrame(rows)
print(df_dataset.to_string(index=False))

    Subset  Imágenes
  Training      3072
Validation       293
      Test       146
     Total      3511


### Tabla 2 — Distribución por clase

In [51]:
dataset_base = Path('/Users/fernando/Desktop/tesis_backup/v2/dataset/ultra')
class_names = {0: 'libre', 1: 'no_disponible', 2: 'ocupado'}
counts = {0: 0, 1: 0, 2: 0}

for subset in ['train', 'valid', 'test']:
    labels_path = dataset_base / subset / 'labels'
    if labels_path.exists():
        for txt in labels_path.glob('*.txt'):
            for line in txt.read_text().splitlines():
                parts = line.strip().split()
                if parts:
                    cls = int(parts[0])
                    if cls in counts:
                        counts[cls] += 1

total = sum(counts.values())

rows = []
for cls_id, name in class_names.items():
    n = counts[cls_id]
    rows.append({'Class': name, 'Instances': n, 'Percentage (%)': round(n / total * 100, 2)})
rows.append({'Class': 'Total', 'Instances': total, 'Percentage (%)': 100.0})

df_classes = pd.DataFrame(rows)
print(df_classes.to_string(index=False))

        Class  Instances  Percentage (%)
        libre      30519           57.95
no_disponible       3511            6.67
      ocupado      18634           35.38
        Total      52664          100.00


### Table 3 Dataset partitioning and augmentation


In [3]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

base = Path('/Users/fernando/Desktop/tesis_backup/v2/modelos/v2')                                        
                                                                                            
MODELS = {                                                                                              
    'YOLOv8s':      'yolov8s_full.json',                                                            
    'YOLOv11s':     'yolov11s_full.json',                                                               
    'YOLOv12s':     'yolov12s_full.json',
    'RT-DETR':      'rtdetr_full.json',                                                                 
    'Faster R-CNN': 'fasterrcnnv2_full.json',                                                           
}                                                                                                       
                                                                                                        
rows = []                                                                                               
                                                                                            
for label, fname in MODELS.items():                                                                     
    path = base / fname
    if not path.exists():                                                                               
        print(f"⚠️ Archivo no encontrado: {fname}")
        continue                                                                                        
                                                                                                    
    with open(path, 'r') as f:                                                                               
        data = json.load(f)                                                                           
                                                                                                        
        # CORRECCIÓN: Todo este bloque debe estar correctamente indentado dentro del bucle
        maps = data['summary']['mAP50']['values']             
                                                                                                        
        n    = len(maps)                                                                                
        mean = np.mean(maps)
        sd   = np.std(maps, ddof=1)
        
        # Intervalo de confianza utilizando la distribución t de Student
        ci   = stats.t.ppf(0.975, df=n - 1) * sd / np.sqrt(n)                                            
      
        row = {'Model': label}                                                                              
        for i, v in enumerate(maps, 1):                                               
            row[f'Run{i}'] = round(v, 4)                                                                 
        
        row['Mean ± SD'] = f'{mean:.4f} ± {sd:.4f}'                                                     
        row['IC 95%']    = f'[{mean - ci:.4f}, {mean + ci:.4f}]'                                           
        rows.append(row)                                                                                
                                                                                                        
# CORRECCIÓN: Se eliminó el bloque duplicado innecesario
if not rows:                                                                                            
    print("❌ No se encontraron datos válidos.")          
else:                                                                                                   
    max_runs = max(len([k for k in r if k.startswith('Run')]) for r in rows)                        
    run_cols = [f'Run{i}' for i in range(1, max_runs + 1)]                                            
                                                                                                        
    df_table = (                                                                                    
        pd.DataFrame(rows)                                                                              
          .reindex(columns=['Model'] + run_cols + ['Mean ± SD', 'IC 95%'])
          .fillna('-')                                                                                
    )                                                                                               
                                                                                                        
    print("\nDetailed Runs Table - mAP@0.5\n")                                                    
    print(df_table.to_string(index=False))
    
    # ── Exportación limpia a LaTeX para tu Tesis ─────────────────────────────────
    print("\n--- Código LaTeX para la Tesis ---")
    # Dinámicamente ajustamos el formato de las columnas según la cantidad de corridas reales
    col_format = 'l' + 'r' * len(run_cols) + 'cc'
    
    latex_output = (
        df_table.style
        .hide(axis='index')
        .to_latex(column_format=col_format, hrules=True)
    )
    print(latex_output)


Detailed Runs Table - mAP@0.5

       Model   Run1   Run2   Run3    Run4    Run5    Run6    Run7    Run8    Run9   Run10       Mean ± SD           IC 95%
     YOLOv8s 0.9947 0.9950 0.9945  0.9946  0.9949  0.9947   0.995   0.995  0.9947  0.9946 0.9948 ± 0.0002 [0.9946, 0.9949]
    YOLOv11s 0.9947 0.9947 0.9946  0.9948  0.9947  0.9947  0.9946  0.9947  0.9947  0.9946 0.9947 ± 0.0001 [0.9946, 0.9947]
    YOLOv12s 0.9945 0.9946 0.9944  0.9944  0.9946  0.9947  0.9946  0.9946  0.9945  0.9946 0.9946 ± 0.0001 [0.9945, 0.9946]
     RT-DETR 0.9947 0.9946 0.9946  0.9946  0.9945  0.9947  0.9947  0.9943  0.9943  0.9947 0.9946 ± 0.0002 [0.9945, 0.9947]
Faster R-CNN 0.9931 0.9931 0.9963       -       -       -       -       -       -       - 0.9942 ± 0.0018 [0.9896, 0.9988]

--- Código LaTeX para la Tesis ---
\begin{tabular}{lrrrrrrrrrrcc}
\toprule
Model & Run1 & Run2 & Run3 & Run4 & Run5 & Run6 & Run7 & Run8 & Run9 & Run10 & Mean ± SD & IC 95% \\
\midrule
YOLOv8s & 0.994700 & 0.995000 & 0.994500 & 0

In [8]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps  # Reemplazo moderno para cm.get_cmap

# ── Configuración de Semillas y Rutas ─────────────────────────────────────────
SEEDS = [42, 123, 456, 789, 1000, 1234, 2024, 2025, 3141, 9999]         
base  = Path('/Users/fernando/Desktop/tesis_backup/v2')            
OUT   = base / 'figuras/convergencia'                                                
OUT.mkdir(parents=True, exist_ok=True)                                             
                                                                                                        
# ── PARTE 1: Modelos basados en archivos CSV (YOLO & RT-DETR) ─────────────────
MODELS_CSV = [                                                                                      
    ('(a) YOLOv8s',   base / 'modelos/v2/yolov8s',  'Blues',   'yolov8s_convergencia'),              
    ('(b) YOLOv11s',  base / 'modelos/v2/yolov11s', 'Greens',  'yolov11s_convergencia'),             
    ('(c) YOLOv12s',  base / 'modelos/v2/yolov12s', 'Oranges', 'yolov12s_convergencia'),             
    ('(d) RT-DETR-L', base / 'modelos/v2/rtdetr',   'Purples', 'rtdetr_convergencia'),               
]                                                                                                       
                                                                                                        
for label, folder, cmap_name, filename in MODELS_CSV:                                                   
    fig, ax = plt.subplots(figsize=(6, 4.5))                                                       
    cmap = colormaps[cmap_name]                                                                 
    colors = [cmap(0.35 + 0.065 * i) for i in range(len(SEEDS))]                                             
    
    for i, seed in enumerate(SEEDS):                                                                 
        csv = folder / f'run_{seed}' / 'results.csv'                                                 
        if not csv.exists():                                                                         
            print(f"⚠️  Falta: {csv}")
            continue                                                                     
        
        df = pd.read_csv(csv)                                                                       
        df.columns = df.columns.str.strip()                                                        
        
        if 'epoch' not in df.columns or df.empty: 
            continue                                                                           
        
        df = df.sort_values('epoch').drop_duplicates('epoch', keep='last').reset_index(drop=True)    
        
        # Graficar asegurando mAP@0.5 y agregando +1 a las épocas para consistencia
        ax.plot(df['epoch'] + 1, df['metrics/mAP50(B)'],                                             
                color=colors[i], linewidth=1.3, alpha=0.85, label=f'seed={seed}')                    
    
    # Estética del gráfico
    ax.set_title(label, fontsize=13, fontweight='bold', pad=8)                                        
    ax.set_xlabel('Época', fontsize=11)                                                         
    ax.set_ylabel('mAP@0.5', fontsize=11)                                                         
    ax.set_ylim(0.980, 1.001)                                                                     
    ax.grid(True, linestyle='--', alpha=0.35)                                                         
    ax.legend(fontsize=7, ncol=2, loc='lower right', framealpha=0.7, handlelength=1.2)                 
    
    plt.tight_layout()                                                                             
    plt.savefig(OUT / f'{filename}.png', bbox_inches='tight', dpi=150)                               
    plt.close()                                                                                   
    print(f"✅ {filename}.png")                                                                     
                                                                                                    
# ── PARTE 2: Faster R-CNN basado en archivo JSON ─────────────────────────────
json_path = base / 'modelos/v2/fasterrcnnv2_full.json' 

if not json_path.exists():
    print(f"❌ No se encontró el archivo de Faster R-CNN en: {json_path}")
else:
    with open(json_path) as f:                                                                               
        frcnn_data = json.load(f)                                                                       
                                                                                                    
    fig, ax = plt.subplots(figsize=(6, 4.5))                                                             
    cmap = colormaps['Reds']                                     
    
    total_frcnn_runs = len(frcnn_data['runs'])
    # CORRECCIÓN: Evitamos IndexError dinámicamente si cambia el número de corridas
    colors = [cmap(0.35 + 0.15 * i) for i in range(total_frcnn_runs)]                                   
                                                                                                    
    for i, run in enumerate(frcnn_data['runs']):                                                         
        seed = run['seed']                                      
        curve = run['map_curve']                                                                                                     
        
        # CORRECCIÓN: Forzamos homogeneidad sumando +1 si el JSON inicia en la época 0
        epochs = [e['epoch'] + 1 for e in curve]                                                         
        map50_vals = [e['mAP50'] for e in curve]                                                       
        
        ax.plot(epochs, map50_vals,                                                                     
                color=colors[i], linewidth=1.3, alpha=0.85, label=f'seed={seed}')
                                                                                                        
    ax.set_title('(e) Faster R-CNN', fontsize=13, fontweight='bold', pad=8)                              
    ax.set_xlabel('Época', fontsize=11)                                                                 
    ax.set_ylabel('mAP@0.5', fontsize=11)                                                                 
    ax.set_ylim(0.980, 1.001)                                                                             
    ax.grid(True, linestyle='--', alpha=0.35)                                                           
    ax.legend(fontsize=7, ncol=1, loc='lower right', framealpha=0.7, handlelength=1.2)                    
    
    plt.tight_layout()                                                                                  
    plt.savefig(OUT / 'fasterrcnn_convergencia.png', bbox_inches='tight', dpi=150)                      
    plt.close()                                                                                         
    print("✅ fasterrcnn_convergencia.png")                                                    
    
print(f"\n📁 Todo el set de gráficos fue guardado con éxito en: {OUT}")

✅ yolov8s_convergencia.png
✅ yolov11s_convergencia.png
✅ yolov12s_convergencia.png
✅ rtdetr_convergencia.png
✅ fasterrcnn_convergencia.png

📁 Todo el set de gráficos fue guardado con éxito en: /Users/fernando/Desktop/tesis_backup/v2/figuras/convergencia


### Table 6. Global detection performance across five architectures (mean ± SD and 95% CI over 10 independent runs).



In [22]:
import json
from pathlib import Path
import numpy as np
from scipy import stats
import pandas as pd  # Usaremos pandas para guardar la tabla limpia

base = Path('/Users/fernando/Desktop/tesis_backup/v2/modelos/v2')                                                                                                
                                                                                                                                                                 
FILES = {                                                                                                                                                        
    'YOLOv8s':      base / 'yolov8s_full.json',                                                                                                             
    'YOLOv11s':     base / 'yolov11s_full.json',                                                                                                            
    'YOLOv12s':     base / 'yolov12s_full.json',                                                                                                            
    'RT-DETR':      base / 'rtdetr_full.json',
    'Faster R-CNN': base / 'fasterrcnnv2_full.json',                                                                                                          
}                                                                                                                                                                
                                                                                                                                                                 
METRICS = ['precision', 'recall', 'f1', 'mAP50', 'mAP50_95']                                                                                                     
HEADERS = ['P', 'R', 'F1', 'mAP@0.5', 'mAP@0.5:0.95']                                                                                                            
                                                                                                                                                                 
def fmt(values):                                                                                                                                                 
    n  = len(values)                                                                                                                                             
    m  = np.mean(values)                                                                                                                                         
    sd = np.std(values, ddof=1)                                                                                                                                  
    ci = stats.t.ppf(0.975, df=n - 1) * sd / np.sqrt(n)                                                                                                          
    return f'{m:.4f} ± {sd:.4f} [{m-ci:.4f}, {m+ci:.4f}]'                                                                                                        
                                                                                                                                                                 
# Diccionario para armar el DataFrame de Pandas
rows_resumen = []

for model, path in FILES.items():                                                                                                                                
    if not path.exists():
        print(f"⚠️ Archivo no encontrado para {model}: {path}")
        continue

    with open(path) as f:                                                                                                                                        
        runs = [r['final'] for r in json.load(f)['runs']]                                                                                                       
                                                                                                                                                                 
    print(f'\n{model}  (n={len(runs)})')                                                                                                                        
    print('-' * 60)                                                                                                                                              
    
    model_row = {'Modelo': model, 'Muestras (n)': len(runs)}
    
    for key, header in zip(METRICS, HEADERS):                                                                                                                    
        vals = [r[key] for r in runs if key in r]                                                                                                               
        formato_texto = fmt(vals)
        print(f'  {header:<15} {formato_texto}')
        
        # Guardamos en el diccionario el formato procesado
        model_row[header] = formato_texto
        
    rows_resumen.append(model_row)

# 📊 Exportación automática de resultados para la tesis
if rows_resumen:
    df = pd.DataFrame(rows_resumen)
    
    # Guardar en CSV para abrir en Excel
    csv_path = base / 'tabla_estadistica_modelos.csv'
    df.to_csv(csv_path, index=False)
    
    print('\n' + '='*60)
    print(f'💾 ¡Resultados consolidados guardados en Excel/CSV!:\n👉 {csv_path}')
    print('='*60)


YOLOv8s  (n=10)
------------------------------------------------------------
  P               0.9964 ± 0.0015 [0.9953, 0.9974]
  R               0.9971 ± 0.0009 [0.9965, 0.9978]
  F1              0.9967 ± 0.0009 [0.9961, 0.9974]
  mAP@0.5         0.9948 ± 0.0002 [0.9946, 0.9949]
  mAP@0.5:0.95    0.9916 ± 0.0003 [0.9913, 0.9918]

YOLOv11s  (n=10)
------------------------------------------------------------
  P               0.9964 ± 0.0009 [0.9958, 0.9970]
  R               0.9975 ± 0.0007 [0.9971, 0.9980]
  F1              0.9970 ± 0.0005 [0.9966, 0.9973]
  mAP@0.5         0.9947 ± 0.0001 [0.9946, 0.9947]
  mAP@0.5:0.95    0.9910 ± 0.0004 [0.9907, 0.9913]

YOLOv12s  (n=10)
------------------------------------------------------------
  P               0.9963 ± 0.0008 [0.9958, 0.9969]
  R               0.9975 ± 0.0006 [0.9971, 0.9979]
  F1              0.9969 ± 0.0004 [0.9967, 0.9972]
  mAP@0.5         0.9946 ± 0.0001 [0.9945, 0.9946]
  mAP@0.5:0.95    0.9906 ± 0.0003 [0.9903, 0.9908]

## §4.3.2 — Per-Class Performance.

In [10]:
import json
from pathlib import Path
import numpy as np
from scipy import stats

base = Path('/Users/fernando/Desktop/tesis_backup/v2/modelos/v2')                                              
                                                                                                               
FILES = {                                                                                                      
    'YOLOv8s':  base / 'yolov8s_full.json',                                                                   
    'YOLOv11s': base / 'yolov11s_full.json',                                                                   
    'YOLOv12s': base / 'yolov12s_full.json',
    'RT-DETR': base / 'rtdetr_full.json',
    'Faster R-CNN': base / 'fasterrcnnv2_full.json', 

}                                                                                                              
                                                                                                               
GLOBAL_METRICS = ['mAP50', 'mAP50_95', 'precision', 'recall', 'f1',                                            
                  'fps', 'lat_p50', 'lat_p95', 'gpu_mb']                                                      
CLASS_METRICS  = ['mAP50', 'mAP50_95', 'precision', 'recall']                                                 
CLASSES        = ['libre', 'ocupado', 'no_disponible']                                                         
                                                                                                               
def ci95(values):                                                                                             
    n    = len(values)                                                                                         
    mean = np.mean(values)                                                                                     
    sd   = np.std(values, ddof=1)
    t_c  = stats.t.ppf(0.975, df=n - 1)                                                                       
    ci   = t_c * sd / np.sqrt(n)
    return mean, sd, ci                                                                                       
                                                                                                               
def fmt(mean, sd, ci):                                                                                         
    return f'{mean:.4f} ± {sd:.4f}  [{mean-ci:.4f}, {mean+ci:.4f}]'                                           
                                                                                                               
for model_name, path in FILES.items():
    with open(path) as f:                                                                                     
        data = json.load(f)                                                                                   

    runs = [r['final'] for r in data['runs']]                                                                 
    n    = len(runs)
    print(f"\n{'='*65}")                                                                                      
    print(f"  {model_name}  (n={n})   t-dist df={n-1}, alpha=0.05")                                            
    print(f"{'='*65}")                                                                                        
                                                                                                               
    print("\n--- Global Metrics ---")                                                                         
    print(f"{'Metric':<15} {'Mean ± SD':<22} 95% CI")                                                         
    print('-' * 65)                                                                                           
    for m in GLOBAL_METRICS:                                                                                  
        vals = [r[m] for r in runs if m in r]                                                                 
        if not vals:                                                                                          
            continue
        mean, sd, ci = ci95(vals)                                                                             
        print(f"{m:<15} {fmt(mean, sd, ci)}")                                                                 
                                                                                                               
    print("\n--- Per-Class Metrics ---")
    for cls in CLASSES:
        print(f"\n  [{cls}]")
        print(f"  {'Metric':<15} {'Mean ± SD':<22} 95% CI")
        print('  ' + '-' * 60)
        for m in CLASS_METRICS:
            vals = [r['per_class'][cls][m] for r in runs
                    if 'per_class' in r and cls in r['per_class'] and m in r['per_class'][cls]]
            if not vals:
                continue
            mean, sd, ci = ci95(vals)
            print(f"  {m:<15} {fmt(mean, sd, ci)}")


  YOLOv8s  (n=10)   t-dist df=9, alpha=0.05

--- Global Metrics ---
Metric          Mean ± SD              95% CI
-----------------------------------------------------------------
mAP50           0.9948 ± 0.0002  [0.9946, 0.9949]
mAP50_95        0.9916 ± 0.0003  [0.9913, 0.9918]
precision       0.9964 ± 0.0015  [0.9953, 0.9974]
recall          0.9971 ± 0.0009  [0.9965, 0.9978]
f1              0.9967 ± 0.0009  [0.9961, 0.9974]
fps             205.4900 ± 6.3776  [200.9277, 210.0523]
lat_p50         4.7680 ± 0.1446  [4.6646, 4.8714]
lat_p95         5.2370 ± 0.1908  [5.1005, 5.3735]
gpu_mb          4093.3000 ± 332.6393  [3855.3442, 4331.2558]

--- Per-Class Metrics ---

  [libre]
  Metric          Mean ± SD              95% CI
  ------------------------------------------------------------
  mAP50           0.9945 ± 0.0005  [0.9941, 0.9948]
  mAP50_95        0.9877 ± 0.0013  [0.9868, 0.9886]
  precision       0.9968 ± 0.0027  [0.9948, 0.9987]
  recall          0.9947 ± 0.0016  [0.9935, 0.9

In [26]:
import json
from pathlib import Path
import pandas as pd

base = Path('/Users/fernando/Desktop/tesis_backup/v2/modelos/v2')                                              

# CORREGIDO: Se eliminó la coma final para que se guarde como dict y no como tuple
MODELS_DATA = {
    'YOLOv8s':  {'json': base / 'yolov8s_full.json',  'folder': base / 'yolov8s'},
    'YOLOv11s': {'json': base / 'yolov11s_full.json', 'folder': base / 'yolov11s'},
    'YOLOv12s': {'json': base / 'yolov12s_full.json', 'folder': base / 'yolov12s'},
    'rt-detr':  {'json': base / 'rtdetr_full.json',  'folder': base / 'rtdetr'}, 
    'fasterrcnn': {'json': base / 'fasterrcnnv2_full.json', 'folder': base / 'fasterrcnn'}
}

# Las semillas que mapean el orden de tus runs en el JSON (Run #1 = semilla 42, etc.)
SEEDS = [42, 123, 456, 789, 1000, 1234, 2024, 2025, 3141, 9999]
TARGET_METRIC = 'mAP50_95'

print(f"\n{'='*85}")
print(f"  ANÁLISIS DE ÉPOCAS CRUZADO (JSON + RESULTS.CSV)")
print(f"{'='*85}\n")

for model_name, info in MODELS_DATA.items():
    json_path = info['json']
    folder_path = info['folder']
    
    if not json_path.exists():
        print(f"Archivo no encontrado: {json_path.name}")
        print('-' * 85)
        continue
        
    with open(json_path) as f:                                                                                     
        data = json.load(f)                                                                                   

    # 1. Identificar el mejor Run según el bloque 'final' del JSON
    best_val = -1.0
    best_run_idx = -1
    
    for idx, r in enumerate(data['runs'], 1):
        final_metrics = r.get('final', {})
        if TARGET_METRIC in final_metrics:
            current_val = final_metrics[TARGET_METRIC]
            if current_val > best_val:
                best_val = current_val
                best_run_idx = idx

    # 2. Si encontramos el mejor run, leemos su CSV para auditar las épocas
    if best_run_idx != -1:
        # Obtenemos la semilla que le corresponde a ese índice de Run
        seed_ganadora = SEEDS[best_run_idx - 1]
        csv_path = folder_path / f'run_{seed_ganadora}' / 'results.csv'
        
        print(f"🏆 MODELO: {model_name}")
        print(f"  -> Ganador en JSON: Run #{best_run_idx} (Semilla {seed_ganadora})")
        print(f"  -> {TARGET_METRIC} Final en JSON: {best_val:.4f}")
        
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df.columns = df.columns.str.strip()
            
            # Buscar columna de época de forma flexible ('epoch' o 'Época')
            epoch_col = [c for c in df.columns if 'epoch' in c.lower() or 'época' in c.lower()]
            
            if not df.empty and epoch_col:
                col_e = epoch_col[0]
                df = df.sort_values(col_e).drop_duplicates(col_e, keep='last').reset_index(drop=True)
                total_epochs = len(df)
                
                # Búsqueda flexible de mAP50_95 en el CSV (tanto formato YOLO como plano para FasterRCNN)
                col_map = [c for c in df.columns if 'map50' in c.lower() and '95' in c]
                
                if col_map:
                    csv_metric_col = col_map[0]
                    idx_max = df[csv_metric_col].idxmax()
                    best_epoch_raw = df.loc[idx_max, col_e]
                    best_epoch_val = df.loc[idx_max, csv_metric_col]
                    
                    # Ajuste si el CSV empieza a contar desde 0
                    exact_best_epoch = best_epoch_raw + 1 if best_epoch_raw < total_epochs else best_epoch_raw
                    
                    print(f"  -> Épocas totales en CSV: {total_epochs}")
                    print(f"  -> 🎯 Época del pico más alto (CSV): Época {int(exact_best_epoch)} (Valor: {best_epoch_val:.4f})")
                else:
                    print(f"  ⚠️ No se encontró la columna mAP50_95 en el CSV. Columnas disponibles: {list(df.columns[:5])}...")
            else:
                print("  ⚠️ El archivo CSV está vacío o no contiene una columna identificable como 'epoch'.")
        else:
            print(f"  ❌ No se encontró el archivo físico de métricas en: {csv_path}")
            
        print('-' * 85)


  ANÁLISIS DE ÉPOCAS CRUZADO (JSON + RESULTS.CSV)

🏆 MODELO: YOLOv8s
  -> Ganador en JSON: Run #8 (Semilla 2025)
  -> mAP50_95 Final en JSON: 0.9920
  -> Épocas totales en CSV: 52
  -> 🎯 Época del pico más alto (CSV): Época 33 (Valor: 0.9896)
-------------------------------------------------------------------------------------
🏆 MODELO: YOLOv11s
  -> Ganador en JSON: Run #2 (Semilla 123)
  -> mAP50_95 Final en JSON: 0.9918
  -> Épocas totales en CSV: 100
  -> 🎯 Época del pico más alto (CSV): Época 82 (Valor: 0.9901)
-------------------------------------------------------------------------------------
🏆 MODELO: YOLOv12s
  -> Ganador en JSON: Run #6 (Semilla 1234)
  -> mAP50_95 Final en JSON: 0.9910
  -> Épocas totales en CSV: 72
  -> 🎯 Época del pico más alto (CSV): Época 53 (Valor: 0.9903)
-------------------------------------------------------------------------------------
🏆 MODELO: rt-detr
  -> Ganador en JSON: Run #2 (Semilla 123)
  -> mAP50_95 Final en JSON: 0.9914
  -> Épocas tot

### Matrices de Confusion

In [21]:
import json                                                                                                                                                      
import numpy as np                                                                                                                                                
import torch                                                                                                                                                      
import torchvision.transforms as T
from pathlib import Path                                                                                                                                          
from torchvision.datasets import CocoDetection
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2                                                                                              
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor                                                                                            
import matplotlib.pyplot as plt                                                                                                                                  
from ultralytics import YOLO                                                                                                                                      
                                                                                                                                                                 
base      = Path('/Users/fernando/Desktop/tesis_backup/v2/modelos/v2')                                                                                                
DATA_YAML = Path('/Users/fernando/Desktop/tesis_backup/v2/dataset/ultra/data.yaml')                                                                                  
COCO_TEST = Path('/Users/fernando/Desktop/tesis_backup/v2/dataset/coco/test')                                                                                        
DEVICE    = 'mps' if torch.backends.mps.is_available() else 'cpu'                                                                                                    
OUT_DIR   = base / 'matrices'                                                                                                                                         
OUT_DIR.mkdir(exist_ok=True)                                                                                                                                      
                                                                                                                                                                 
# CORREGIDO: Mismo orden que en tus gráficos Lollipop para consistencia en la tesis
CLASSES      = ['libre', 'no_disponible', 'ocupado']
CLASS_LABELS = ['Libre', 'No disponible', 'Ocupado']
NUM_CLASS = 4   # 3 clases + fondo (FasterRCNN)                                                                                                                   
                                                                                                                                                                 
# Modelos ultralytics (YOLO + RT-DETR)                                                                                                                           
ULTRALYTICS = {                                                                                                                                                  
    'YOLOv8s':  base / 'yolov8s'  / 'run_2025' / 'weights' / 'best.pt',                                                                                                
    'YOLOv11s': base / 'yolov11s' / 'run_123'  / 'weights' / 'best.pt',                                                                                                
    'YOLOv12s': base / 'yolov12s' / 'run_1234' / 'weights' / 'best.pt',                                                                                                
    'RT-DETR':  base / 'rtdetr'   / 'run_123'  / 'weights' / 'best.pt',  
                                                                                                 
}                                                                                                                                                                
                                                                                                                                                                 
# ── Función de plot original (Estilizada para Tesis) ───────────────────────────                                                                                  
def plot_cm(matrix, classes_display, model_name, save_path):

    norm = matrix.astype(float) / matrix.sum(axis=1, keepdims=True).clip(min=1)
    norm = np.nan_to_num(norm)

    # Figura más compacta
    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    fig.patch.set_facecolor('white')

    im = ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)

    cbar = plt.colorbar(
        im,
        ax=ax,
        fraction=0.04,
        pad=0.02
    )
    cbar.ax.tick_params(labelsize=8)

    ax.set_xticks(range(len(classes_display)))
    ax.set_yticks(range(len(classes_display)))

    ax.set_xticklabels(
        classes_display,
        rotation=25,
        ha='right',
        fontsize=8.5
    )

    ax.set_yticklabels(
        classes_display,
        fontsize=8.5
    )

    ax.set_xlabel(
        'Predicted',
        fontsize=9.5,
        fontweight='semibold'
    )

    ax.set_ylabel(
        'True',
        fontsize=9.5,
        fontweight='semibold'
    )

    ax.set_title(
        f'Confusion Matrix — {model_name}\n(normalized by row)',
        fontsize=10,
        fontweight='bold',
        pad=6
    )

    for i in range(len(classes_display)):
        for j in range(len(classes_display)):

            color = 'white' if norm[i, j] > 0.55 else 'black'

            ax.text(
                j,
                i,
                f'{norm[i,j]:.3f}\n({int(matrix[i,j])})',
                ha='center',
                va='center',
                fontsize=7.5,
                fontweight='bold',
                color=color
            )

    plt.tight_layout(pad=0.2)

    plt.savefig(
        save_path,
        dpi=400,
        bbox_inches='tight',
        pad_inches=0.02,
        facecolor='white'
    )

    plt.close()

    print(f'   Guardada: {save_path.name}')                                                                                                                  
                                                                                                                                                                 
                                                                                                                                                                 
# ── 1. Ultralytics (YOLOv8s / v11s / v12s / RT-DETR) ─────────────────────────
for model_name, weights in ULTRALYTICS.items():                                                                                                                  
    if not weights.exists():                                                                                                                                     
        print(f'\n{model_name}: best.pt no encontrado → {weights}')                                                                                                    
        continue                                                                                                                                                 
    print(f'\nEvaluando {model_name}...')                                                                                                                        
    model   = YOLO(str(weights))
    results = model.val(data=str(DATA_YAML), split='test',                                                                                                       
                        verbose=False, device=DEVICE)                                                                                                            
    matrix  = results.confusion_matrix.matrix.astype(int)                                                                                                        
    nc = len(CLASSES)                                                                                                                                            
    if matrix.shape[0] == nc + 1:       
        matrix = matrix[:nc, :nc]                                                                                                                                
    save_path = OUT_DIR / f'confusion_{model_name.lower().replace("-","_").replace(" ","_")}.png'                                                                    
    plot_cm(matrix, CLASS_LABELS, model_name, save_path)                                                                                                          
                                                                                                                                                                 
                                                                                                                                                                 
# ── 2. Faster R-CNN (evaluación manual con PyTorch) ───────────────────────────                                                                                 
def iou(box_a, box_b):                                                                                                                                           
    """box: [x1,y1,x2,y2]"""                                                                                                                                     
    xi1 = max(box_a[0], box_b[0]); yi1 = max(box_a[1], box_b[1])                                                                                                 
    xi2 = min(box_a[2], box_b[2]); yi2 = min(box_a[3], box_b[3])                                                                                                 
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)                                                                                                                
    area_a = (box_a[2]-box_a[0]) * (box_a[3]-box_a[1])                                                                                                           
    area_b = (box_b[2]-box_b[0]) * (box_b[3]-box_b[1])                                                                                                           
    return inter / (area_a + area_b - inter + 1e-6)                                                                                                              
                                                                                                                                                                 
def fasterrcnn_confusion(weights_path, coco_test_dir, classes,                                                                                                   
                         iou_thresh=0.5, score_thresh=0.5, device='cpu'):                                                                                        
    nc = len(classes)                                                                                                                                            
    cm = np.zeros((nc, nc), dtype=int)                                                                                                                           
                                                                                                                                                                 
    dev = torch.device(device)                                                                                                                                   
    model = fasterrcnn_resnet50_fpn_v2(weights=None)
    in_f  = model.roi_heads.box_predictor.cls_score.in_features                                                                                                  
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, NUM_CLASS)                                                                                            
    model.load_state_dict(torch.load(weights_path, map_location=dev))                                                                                            
    model.to(dev); model.eval()         
                                                                                                                                                                 
    ds = CocoDetection(                                                                                                                                          
        root=str(coco_test_dir),                                                                                                                                 
        annFile=str(coco_test_dir / '_annotations.coco.json')                                                                                                    
    )                                                                                                                                                            
    coco_cats = ds.coco.loadCats(ds.coco.getCatIds())
    cat_id_to_idx = {c['id']: classes.index(c['name']) for c in coco_cats                                                                                        
                     if c['name'] in classes}                                                                                                                    
                                                                                                                                                                 
    with torch.no_grad():                                                                                                                                        
        for img_pil, anns in ds:                                                                                                                                 
            img_t = T.ToTensor()(img_pil).to(dev)                                                                                                                
            preds = model([img_t])[0]                                                                                                                            
                                                                                                                                                                 
            gt_boxes  = [[a['bbox'][0], a['bbox'][1],                                                                                                             
                          a['bbox'][0]+a['bbox'][2],                                                                                                             
                          a['bbox'][1]+a['bbox'][3]] for a in anns if a['bbox'][2]>0]                                                                            
            gt_labels = [cat_id_to_idx[a['category_id']] for a in anns                                                                                           
                         if a['bbox'][2]>0 and a['category_id'] in cat_id_to_idx]                                                                                
                                                                                                                                                                 
            keep   = preds['scores'] > score_thresh                                                                                                              
            p_boxes  = preds['boxes'][keep].cpu().tolist()                                                                                                       
            p_labels = preds['labels'][keep].cpu().tolist()                                                                                                      
                                                                                                                                                                 
            matched_gt = set()                                                                                                                                   
            for pb, pl in zip(p_boxes, p_labels):                                                                                                                
                pred_idx = pl - 1   # FasterRCNN: 1-indexed → 0-indexed
                if pred_idx < 0 or pred_idx >= nc:                                                                                                               
                    continue                                                                                                                                     
                best_iou, best_j = 0, -1                                                                                                                         
                for j, gb in enumerate(gt_boxes):                                                                                                                
                    if j in matched_gt:                                                                                                                          
                        continue                                                                                                                                 
                    v = iou(pb, gb)                                                                                                                              
                    if v > best_iou:                                                                                                                             
                        best_iou, best_j = v, j
                if best_iou >= iou_thresh and best_j >= 0:                                                                                                       
                    matched_gt.add(best_j)                                                                                                                       
                    gt_idx = gt_labels[best_j]                                                                                                                   
                    cm[gt_idx, pred_idx] += 1                                                                                                                    
                                                                                                                                                                 
    return cm                                                                                                                                                    
                                                                                                                                                                 
weights_frcnn = base / 'fasterrcnnv2' / 'best_seed456.pt'                                                                                                           
if weights_frcnn.exists():                                                                                                                                       
    print('\nEvaluando Faster R-CNN...')
    cm = fasterrcnn_confusion(weights_frcnn, COCO_TEST, CLASSES, device=DEVICE)                                                                                  
    plot_cm(cm, CLASS_LABELS, 'Faster R-CNN', OUT_DIR / 'confusion_faster_rcnn.png')                                                                             
else:                                                                                                                                                            
    print(f'\nFaster R-CNN: peso no encontrado → {weights_frcnn}')                                                                                               
                                                                                                                                                                 
print(f'\nTodas las matrices guardadas en: {OUT_DIR}')


Evaluando YOLOv8s...
Ultralytics 8.4.56 🚀 Python-3.13.3 torch-2.12.0 MPS (Apple M4)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 261.6±50.4 MB/s, size: 91.6 KB)
val: Scanning /Users/fernando/Desktop/tesis_backup/v2/dataset/ultra/test/labels.cache... 146 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 146/146 563.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.2it/s 8.4s.8ss
                   all        146       2190      0.999      0.998      0.995      0.992
Speed: 0.4ms preprocess, 25.8ms inference, 0.0ms loss, 13.7ms postprocess per image
Results saved to /Users/fernando/Desktop/tesis_backup/v2/modelos/v2/runs/detect/val-80
   Guardada: confusion_yolov8s.png

Evaluando YOLOv11s...
Ultralytics 8.4.56 🚀 Python-3.13.3 torch-2.12.0 MPS (Apple M4)
YOLO11s summary (fused): 101 layers, 9,413,961 parameters, 0 gradients

### IGURA 3 — Gráfica de barras por clase


In [17]:
import json                                                                                                                                                      
from pathlib import Path
import matplotlib.pyplot as plt                                                                                                                                  
import numpy as np                                                                                                                                                
from scipy import stats                                                                                                                                          
                                                                                                                                                                 
base = Path('/Users/fernando/Desktop/tesis_backup/v2/modelos/v2')                                                                                                
                                                                                                                                                                 
FILES = {                                                                                                                                                        
    'YOLOv8s':      base / 'yolov8s_full.json',                                                                                                             
    'YOLOv11s':     base / 'yolov11s_full.json',
    'YOLOv12s':     base / 'yolov12s_full.json',                                                                                                            
    'RT-DETR':      base / 'rtdetr_full.json',                                                                                                              
    'Faster R-CNN': base / 'fasterrcnn_full.json',                                                                                                          
}                                                                                                                                                                
                                                                                                                                                                 
CLASSES      = ['libre', 'ocupado', 'no_disponible']                                                                                                             
CLASS_LABELS = ['Libre', 'Ocupado', 'No disponible']                                                                                                             
METRIC       = 'mAP50'                                                                                                                                           
                                                                                                                                                                 
PALETTE = {                                                                                                                                                      
    'libre':          '#2563EB',                                                                                                                                 
    'ocupado':        '#DC2626',                                                                                                                                 
    'no_disponible':  '#16A34A',                                                                                                                                 
}                                                                                                                                                                
                                                                                                                                                                 
def ci95(values):                                                                                                                                                
    n  = len(values)
    m  = np.mean(values)                                                                                                                                         
    sd = np.std(values, ddof=1)                                                                                                                                  
    ci = stats.t.ppf(0.975, df=n - 1) * sd / np.sqrt(n)                                                                                                          
    return m, ci                                                                                                                                                 
                                                                                                                                                                 
model_names = list(FILES.keys())                                                                                                                                 
means  = {c: [] for c in CLASSES}                                                                                                                                
errors = {c: [] for c in CLASSES}                                                                                                                                
                                                                                                                                                                 
# Extraer datos de todos los archivos
for model, path in FILES.items():                                                                                                                                
    if not path.exists():
        print(f"⚠️ Archivo no encontrado para {model}, saltando...")
        continue
        
    with open(path) as f:                                                                                                                                        
        runs = [r['final'] for r in json.load(f)['runs']]
        
    for cls in CLASSES:                                                                                                                                             
        vals = [r['per_class'][cls][METRIC] for r in runs if 'per_class' in r and cls in r['per_class']]                                                            
        
        if vals:
            m, ci = ci95(vals)                                                                                                                                          
            means[cls].append(m)                                                                                                                                        
            errors[cls].append(ci)                                                                                                                                      
        else:
            means[cls].append(0)
            errors[cls].append(0)

# ── Generar Figuras Individuales por Clase ─────────────────────────────────
y = np.arange(len(model_names))                                                                                                                                  

for cls, label in zip(CLASSES, CLASS_LABELS):
    # Crear una figura independiente para cada clase
    fig, ax = plt.subplots(figsize=(6, 4.5))
    fig.patch.set_facecolor('white')
    
    color = PALETTE[cls]                                                                                                                                         
    ms    = np.array(means[cls])                                                                                                                                 
    es    = np.array(errors[cls])                                                                                                                                 
                                                                                                                                                                 
    # Línea de referencia (Promedio general de la clase)
    ax.axvline(x=ms.mean(), color=color, linewidth=0.8, linestyle='--', alpha=0.4, zorder=1)                                                                                                                                                    
                                                                                                                                                                 
    # Lollipop lines
    for i in range(len(model_names)):                                                                                                                            
        ax.plot([ms.mean(), ms[i]], [y[i], y[i]], color='#CBD5E1', linewidth=1.5, zorder=2)                                                                                                                     
                                                                                                                                                                 
    # Barras de error (IC 95%)                                                                                                                                   
    ax.errorbar(ms, y, xerr=es, fmt='none', ecolor=color, elinewidth=1.5, capsize=4, capthick=1.5, zorder=3)                                                                                                                   
                                                                                                                                                                 
    # Marcador principal                                                                                                                                         
    ax.scatter(ms, y, s=120, color=color, zorder=4, edgecolors='white', linewidths=1.2)                                                                          
                                                                                                                                                                 
    # Anotación del valor                                                                                                                                          
    for i, (m, e) in enumerate(zip(ms, es)):                                                                                                                     
        ax.text(m + e + 0.0015, y[i], f'{m:.4f}', va='center', ha='left', fontsize=9.5, color='#1E293B')                                                                                                               
                                                                                                                                                                 
    ax.set_yticks(y)                                                                                                                                             
    ax.set_yticklabels(model_names, fontsize=10.5)                                                                                                                 
    ax.set_xlabel('mAP@0.5', fontsize=11)                                                                                                                       
    ax.set_title(f'Rendimiento Clase: {label}\n' r'(mean ± 95% CI, $n$=10)', fontsize=12, fontweight='bold', pad=12, color=color)                                                                                     
                                                                                                                                                                 
    # Rango dinámico
    margin = max(es) * 4 if max(es) > 0 else 0.05
    ax.set_xlim(ms.min() - margin, ms.max() + margin + 0.005)                                                                                                     
                                                                                                                                                                 
    ax.spines[['top', 'right']].set_visible(False)                                                                                                               
    ax.spines['left'].set_color('#E2E8F0')                                                                                                                       
    ax.spines['bottom'].set_color('#E2E8F0')                                                                                                                     
    ax.tick_params(axis='both', which='both', length=0)                                                                                                          
    ax.grid(axis='x', linestyle='--', alpha=0.4, color='#E2E8F0')                                                                                                
    ax.set_facecolor('#FAFAFA')                                                                                                                                  
                                                                                                                                                                 
    plt.tight_layout()                                                                                                                                               
    
    # Nombre único para cada archivo de salida
    output_name = f'figura3_per_class_{cls}.png'
    save_path = base / output_name
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')                                                                                                    
    plt.close() # Cierra la figura actual para liberar memoria antes de la siguiente iteración
    
    print(f'✅ Guardada: {save_path}')

✅ Guardada: /Users/fernando/Desktop/tesis_backup/v2/modelos/v2/figura3_per_class_libre.png
✅ Guardada: /Users/fernando/Desktop/tesis_backup/v2/modelos/v2/figura3_per_class_ocupado.png
✅ Guardada: /Users/fernando/Desktop/tesis_backup/v2/modelos/v2/figura3_per_class_no_disponible.png


siguente

In [13]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

base = Path('/Users/fernando/Desktop/tesis_backup/v2/modelos/v2')
FILES = {
    'YOLOv8s':  base / 'yolov8s_full.json',
    'YOLOv11s': base / 'yolov11s_full.json',
    'YOLOv12s': base / 'yolov12s_full.json',
    'RT-DETR':      base / 'rtdetr_full.json',                                                                                                              
    'Faster R-CNN': base / 'fasterrcnnv2_full.json',  
}

METRICS = {
    'FPS':       'fps',
    'Lat p50':   'lat_p50',
    'Lat p95':   'lat_p95',
    'GPU MB':    'gpu_mb',
}

def ci95(values):
    n  = len(values)
    m  = np.mean(values)
    sd = np.std(values, ddof=1)
    ci = stats.t.ppf(0.975, df=n-1) * sd / np.sqrt(n)
    return m, sd, ci

rows = []
for model_name, path in FILES.items():
    with open(path) as f:
        runs = [r['final'] for r in json.load(f)['runs']]
    row = {'Model': model_name, 'n': len(runs)}
    for col, key in METRICS.items():
        vals = [r[key] for r in runs if key in r]
        m, sd, ci = ci95(vals)
        row[col] = f'{m:.2f} ± {sd:.2f}  [{m-ci:.2f}, {m+ci:.2f}]'
    rows.append(row)

# Tabla compacta estilo paper
df = pd.DataFrame(rows)
print(df.to_string(index=False))

print("\n\n--- Formato Sensors MDPI ---")
print(f"{'Model':<12} {'FPS [IC95%]':<30} {'Lat p50 [IC95%]':<28} {'Lat p95 [IC95%]':<28} {'GPU MB [IC95%]'}")
print('-' * 120)
for r in rows:
    print(f"{r['Model']:<12} {r['FPS']:<30} {r['Lat p50']:<28} {r['Lat p95']:<28} {r['GPU MB']}")

       Model  n                             FPS                      Lat p50                      Lat p95                                GPU MB
     YOLOv8s 10 205.49 ± 6.38  [200.93, 210.05]    4.77 ± 0.14  [4.66, 4.87]    5.24 ± 0.19  [5.10, 5.37]  4093.30 ± 332.64  [3855.34, 4331.26]
    YOLOv11s 10 161.20 ± 5.74  [157.09, 165.31]    6.12 ± 0.20  [5.98, 6.26]    6.83 ± 0.85  [6.23, 7.44]    4306.80 ± 2.82  [4304.78, 4308.82]
    YOLOv12s 10  94.24 ± 26.85  [75.03, 113.45]  11.85 ± 4.68  [8.50, 15.20]  12.84 ± 5.61  [8.83, 16.85] 4674.50 ± 1608.97  [3523.51, 5825.49]
     RT-DETR 10    41.07 ± 0.62  [40.63, 41.51] 24.19 ± 0.40  [23.91, 24.48] 25.53 ± 1.05  [24.78, 26.28] 6216.30 ± 1258.70  [5315.88, 7116.72]
Faster R-CNN  3    27.30 ± 0.52  [26.01, 28.59] 36.68 ± 0.71  [34.91, 38.45] 37.05 ± 0.86  [34.91, 39.19]    2965.33 ± 3.06  [2957.74, 2972.92]


--- Formato Sensors MDPI ---
Model        FPS [IC95%]                    Lat p50 [IC95%]              Lat p95 [IC95%]              GPU

confirmacion

### Table 5. Computational efficiency metrics across five architectures (mean ± SD and 95% CI over 10 independent runs).

In [14]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

base = Path('/Users/fernando/Desktop/tesis_backup/v2/modelos/v2')                                         
FILES = {                                                                                  
    'YOLOv8s':  base / 'yolov8s_full.json',                                                
    'YOLOv11s': base / 'yolov11s_full.json',                                               
    'YOLOv12s': base / 'yolov12s_full.json',                                               
    'RT-DETR':  base / 'rtdetr_full.json',
    'fasterrcnn': base / 'fasterrcnnv2_full.json',                                          
}                                                                                          
                                                                                           
METRICS = {                                                                                
    'FPS':          'fps',
    'Lat p50 (ms)': 'lat_p50',                                                             
    'Lat p95 (ms)': 'lat_p95',                                                             
    'GPU MB':       'gpu_mb',
}                                                                                          
                                                                                           
def fmt(values):                                                                           
    n  = len(values)
    m  = np.mean(values)                                                                   
    sd = np.std(values, ddof=1)
    ci = stats.t.ppf(0.975, df=n-1) * sd / np.sqrt(n)                                    
    return f'{m:.2f} ± {sd:.2f} [{m-ci:.2f}, {m+ci:.2f}]'                                  
   
rows = []        
for model, path in FILES.items():                                                          
    if not path.exists():
        print(f"⚠️ Advertencia: Archivo no encontrado para {model} -> {path.name}")
        continue
        
    with open(path) as f:
        runs = [r['final'] for r in json.load(f)['runs']]                                  
        
    row = {'Model': f'{model} (n={len(runs)})'}
    for col, key in METRICS.items():                                                       
        vals = [r[key] for r in runs if key in r]
        if not vals:
            row[col] = "N/A"
            continue
        row[col] = fmt(vals)                                                               
    rows.append(row)                                                                       
                                                                                           
df = pd.DataFrame(rows).set_index('Model')                                                 
print("\nTable 5. Computational efficiency metrics (mean ± SD [95% CI])\n")                
print(df.to_string())


Table 5. Computational efficiency metrics (mean ± SD [95% CI])

                                             FPS                 Lat p50 (ms)                 Lat p95 (ms)                                GPU MB
Model                                                                                                                                           
YOLOv8s (n=10)    205.49 ± 6.38 [200.93, 210.05]     4.77 ± 0.14 [4.66, 4.87]     5.24 ± 0.19 [5.10, 5.37]   4093.30 ± 332.64 [3855.34, 4331.26]
YOLOv11s (n=10)   161.20 ± 5.74 [157.09, 165.31]     6.12 ± 0.20 [5.98, 6.26]     6.83 ± 0.85 [6.23, 7.44]     4306.80 ± 2.82 [4304.78, 4308.82]
YOLOv12s (n=10)    94.24 ± 26.85 [75.03, 113.45]   11.85 ± 4.68 [8.50, 15.20]   12.84 ± 5.61 [8.83, 16.85]  4674.50 ± 1608.97 [3523.51, 5825.49]
RT-DETR (n=10)       41.07 ± 0.62 [40.63, 41.51]  24.19 ± 0.40 [23.91, 24.48]  25.53 ± 1.05 [24.78, 26.28]  6216.30 ± 1258.70 [5315.88, 7116.72]
fasterrcnn (n=3)     27.30 ± 0.52 [26.01, 28.59]  36.68 ± 0.71 [3

### Table 6. Pairwise FPS comparisons (Dunn–Bonferroni post-hoc test).


In [15]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scikit_posthocs import posthoc_dunn
from scipy import stats

base = Path('/Users/fernando/Desktop/tesis_backup/v2/modelos/v2')
FILES = {
    'YOLOv8s':  base / 'yolov8s_full.json',
    'YOLOv11s': base / 'yolov11s_full.json',
    'YOLOv12s': base / 'yolov12s_full.json',
    'RT-DETR':  base / 'rtdetr_full.json',
    'Faster R-CNN': base / 'fasterrcnnv2_full.json',}

METRIC = 'fps'  # Cambia a 'mAP50_95', 'lat_p95', etc. para otras métricas

groups = {}
for model, path in FILES.items():
    if not path.exists():
        print(f"⚠️ Advertencia: Archivo no encontrado para {model} -> {path.name}")
        continue
        
    with open(path) as f:
        runs = [r['final'] for r in json.load(f)['runs']]
    groups[model] = [r[METRIC] for r in runs if METRIC in r]

if not groups or any(len(v) == 0 for v in groups.values()):
    print("❌ Error: Uno o más grupos no contienen datos válidos para procesar.")
else:
    # Kruskal-Wallis
    H, p = stats.kruskal(*groups.values())
    print(f"Kruskal-Wallis  H = {H:.4f}   p = {p:.4f}")
    print(f"{'Significativo' if p < 0.05 else 'No significativo'} (α=0.05)\n")
    
    # Post-hoc Dunn con Bonferroni
    labels = list(groups.keys())
    data = [groups[m] for m in labels]
    
    dunn = posthoc_dunn(data, p_adjust='bonferroni')
    dunn.index = labels
    dunn.columns = labels
    
    print("Post-hoc Dunn (Bonferroni):")
    print(dunn.round(4).to_string())

Kruskal-Wallis  H = 39.9003   p = 0.0000
Significativo (α=0.05)

Post-hoc Dunn (Bonferroni):
              YOLOv8s  YOLOv11s  YOLOv12s  RT-DETR  Faster R-CNN
YOLOv8s        1.0000    0.7492    0.0037   0.0000        0.0001
YOLOv11s       0.7492    1.0000    0.7492   0.0037        0.0134
YOLOv12s       0.0037    0.7492    1.0000   0.7492        0.4590
RT-DETR        0.0000    0.0037    0.7492   1.0000        1.0000
Faster R-CNN   0.0001    0.0134    0.4590   1.0000        1.0000


### §4.3.4 Optimal Model Selection?

In [16]:
import json
from pathlib import Path
import numpy as np

base = Path('/Users/fernando/Desktop/tesis_backup/v2/modelos/v2')
FILES = {
    'YOLOv8s':  base / 'yolov8s_full.json',
    'YOLOv11s': base / 'yolov11s_full.json',
    'YOLOv12s': base / 'yolov12s_full.json',
    'RT-DETR':  base / 'rtdetr_full.json',
    'Faster R-CNN': base / 'fasterrcnnv2_full.json',}

WEIGHTS = {'mAP50_95': 0.60, 'fps': 0.25, 'lat_p95': 0.15}

# Paso 1: medias desde JSON
data = {}
for model, path in FILES.items():
    if not path.exists():
        print(f"⚠️ Advertencia: Archivo no encontrado para {model} -> {path.name}")
        continue
        
    with open(path) as f:
        runs = [r['final'] for r in json.load(f)['runs']]
    
    n = len(runs)
    data[model] = {
        'n':        n,
        'mAP50_95': np.mean([r['mAP50_95'] for r in runs]),
        'fps':      np.mean([r['fps']       for r in runs]),
        'lat_p95':  np.mean([r['lat_p95']   for r in runs]),
    }

models = list(data.keys())

# Paso 2: Min-Max normalization
def minmax(values):
    if not values:
        return []
    mn, mx = min(values), max(values)
    return [(v - mn) / (mx - mn) if mx != mn else 1.0 for v in values]

if not models:
    print("❌ Error: No se encontraron datos válidos para procesar el Score S.")
else:
    map_norm = minmax([data[m]['mAP50_95'] for m in models])
    fps_norm = minmax([data[m]['fps']      for m in models])
    lat_norm = minmax([data[m]['lat_p95']  for m in models])

    # Paso 3: Score S y ranking
    scores = []
    for i, m in enumerate(models):
        s = (WEIGHTS['mAP50_95'] * map_norm[i]
             + WEIGHTS['fps']      * fps_norm[i]
             + WEIGHTS['lat_p95']  * (1 - lat_norm[i]))
        scores.append({
            'Model':      f"{m} (n={data[m]['n']})",
            'mAP95 raw':  data[m]['mAP50_95'],
            'FPS raw':    data[m]['fps'],
            'Lat95 raw':  data[m]['lat_p95'],
            'mAP norm':   map_norm[i],
            'FPS norm':   fps_norm[i],
            '1-Lat norm': 1 - lat_norm[i],
            'Score S':    s,
        })

    scores.sort(key=lambda x: x['Score S'], reverse=True)

    # Tabla resumen
    print("§4.3.4 — Optimal Model Selection\n")
    print(f"{'Model':<18} {'mAP95':>7} {'FPS':>7} {'Lat95':>7}  "
          f"{'mAP norm':>9} {'FPS norm':>9} {'1-Lat':>7} {'Score S':>9}  Rank")
    print('─' * 85)
    for rank, r in enumerate(scores, 1):
        print(f"{r['Model']:<18} {r['mAP95 raw']:>7.4f} {r['FPS raw']:>7.2f} "
              f"{r['Lat95 raw']:>7.2f}  {r['mAP norm']:>9.4f} {r['FPS norm']:>9.4f} "
              f"{r['1-Lat norm']:>7.4f} {r['Score S']:>9.4f}  #{rank}")

    print(f"\nWeights: mAP@0.5:0.95={WEIGHTS['mAP50_95']}  "
          f"FPS={WEIGHTS['fps']}  (1-Lat_p95)={WEIGHTS['lat_p95']}")
    print("Normalization: Min-Max across all models")

§4.3.4 — Optimal Model Selection

Model                mAP95     FPS   Lat95   mAP norm  FPS norm   1-Lat   Score S  Rank
─────────────────────────────────────────────────────────────────────────────────────
YOLOv8s (n=10)      0.9916  205.49    5.24     1.0000    1.0000  1.0000    1.0000  #1
YOLOv11s (n=10)     0.9910  161.20    6.83     0.8415    0.7514  0.9498    0.8352  #2
YOLOv12s (n=10)     0.9906   94.24   12.84     0.7061    0.3757  0.7611    0.6317  #3
RT-DETR (n=10)      0.9907   41.07   25.53     0.7579    0.0773  0.3622    0.5284  #4
Faster R-CNN (n=3)  0.9881   27.30   37.05     0.0000    0.0000  0.0000    0.0000  #5

Weights: mAP@0.5:0.95=0.6  FPS=0.25  (1-Lat_p95)=0.15
Normalization: Min-Max across all models


In [9]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

base = Path('/Users/fernando/Desktop/tesis_backup/v2/modelos/v2')
curves_path = base / 'fasterrcnn_loss_curves.json'

if not curves_path.exists():
    print(f"❌ Error: No se encontró el archivo de curvas en: {curves_path}")
else:
    with open(curves_path) as f:
        curves = json.load(f)
        
    SEEDS = [42, 123, 456, 789, 1000, 1234, 2024, 2025, 3141, 9999]
    
    # Generación de colores limpia usando la API moderna de Matplotlib
    colors = [plt.colormaps['Reds'](0.35 + 0.065 * i) for i in range(10)]
    
    fig, ax = plt.subplots(figsize=(6, 4.5))
    
    for i, seed in enumerate(SEEDS):
        seed_str = str(seed)
        if seed_str not in curves:
            print(f"⚠️ Advertencia: La semilla {seed} no está en el archivo JSON.")
            continue
            
        epochs = [e['epoch'] for e in curves[seed_str]]
        losses = [e['loss']  for e in curves[seed_str]]
        
        ax.plot(epochs, losses, color=colors[i], alpha=0.85, label=f'seed={seed}')
        
    ax.set_title('(e) Faster R-CNN', fontsize=13, fontweight='bold', pad=8)
    ax.set_xlabel('Época', fontsize=11)
    ax.set_ylabel('Training Loss', fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.35)
    ax.legend(fontsize=7, ncol=2, loc='upper right', framealpha=0.7, handlelength=1.2)
    
    plt.tight_layout()
    plt.savefig('fasterrcnn_convergencia.png', bbox_inches='tight', dpi=150)
    plt.close()
    
    print('Guardada con éxito (y sin warnings): fasterrcnn_convergencia.png')

Guardada con éxito (y sin warnings): fasterrcnn_convergencia.png
